# Step 4 — Forecast Dec 11 → Dec 31, 2026 from a newer origin (no retraining)

The trained TFT is reused as is. **Origin** = last complete actual day. The encoder reads the 365 days ending on the origin and the decoder predicts `n = FORECAST_END − ORIGIN` days (60 for a Nov 1 origin, so no autoregression). Rows from `KEEP_FROM` onward are the deliverable.

**You must supply before running**
1. Refreshed actuals through `ORIGIN` in `local_train_data_refresh/chunk_*.parquet` (same layout as Step 1).
2. A calendar with all future covariates extended to Dec 31 (festival flags can't be forward-filled; spend/petrol need your assumed values).
3. `ORIGIN` = last day on which every series is complete.

In [ ]:
import os, json, gc, collections.abc
import numpy as np
import pandas as pd
import torch
from darts import TimeSeries
from darts.models import TFTModel
from pytorch_lightning import Callback

## 1. Config

In [ ]:
BASE = r"C:\Users\G0004878\Desktop\TFT_Data\Daily_forecasting_model\Iterations in October\_1_Data preparation"
MODEL_NAME = "daily_tft_negbin_emb_2026-10-09_11_37_06"      # the model you already trained

ORIGIN        = pd.Timestamp("2026-11-01")      # last COMPLETE actual day, same for all series
FORECAST_END  = pd.Timestamp("2026-12-31")
KEEP_FROM     = pd.Timestamp("2026-12-11")      # rows you actually want
NUM_SAMPLES   = 100                             # NegBin is probabilistic; mean/quantiles come from samples
QUANTILES     = [0.45, 0.55, 0.70, 0.75]        # same ones as your PRED_Q45 ... PRED_Q75 sheet
SERIES_BATCH  = 1500                            # series per predict() call (lower if GPU memory is tight)

# REFRESHED actuals: same chunk_*.parquet layout as Step 1, but pulled through ORIGIN.
# Columns needed: time_col, group_col, target_col and every static covariate.
ACTUALS_DIR   = os.path.join(BASE, "local_train_data_refresh")
# Calendar with ALL future covariates, extended to FORECAST_END (Step 1 output, new file).
CALENDAR_PATH = os.path.join(BASE, "calendar_extended_to_dec31.parquet")
OUT_DIR       = os.path.join(BASE, "forecast_new_origin")

## 2. Load Model

In [ ]:
# The checkpoint pickled your callbacks; a fresh kernel needs a class with the same name.
class TBDiagnostics(Callback):
    def __init__(self, *a, **k): pass

try:                                   # same module you used when training
    import tft_checks                  # noqa: F401  (EmbeddingGate must be importable)
except ImportError:
    pass


def load_model():
    m = TFTModel.load_from_checkpoint(MODEL_NAME, best=True, map_location="cpu")
    return m

## 3. Build Inputs

In [ ]:
def load_roles():
    with open(os.path.join(BASE, "column_roles.json")) as f:
        roles = json.load(f)
    with open(os.path.join(BASE, roles["vocab_path"])) as f:
        vocab = json.load(f)
    return roles, vocab

In [ ]:
def load_calendar(roles, input_len, n):
    time_col, fut = roles["time_col"], roles["future_covariates"]
    cal = pd.read_parquet(CALENDAR_PATH)
    cal[time_col] = pd.to_datetime(cal[time_col])
    cal = cal.sort_values(time_col).reset_index(drop=True)
    need_from = ORIGIN - pd.Timedelta(days=input_len - 1)
    need_to = ORIGIN + pd.Timedelta(days=n)
    assert cal[time_col].min() <= need_from and cal[time_col].max() >= need_to, (
        f"Calendar spans {cal[time_col].min().date()} -> {cal[time_col].max().date()} but must span "
        f"{need_from.date()} -> {need_to.date()}.")
    missing = [c for c in fut if c not in cal.columns]
    assert not missing, f"calendar is missing future covariates: {missing}"
    assert not cal[fut].isna().any().any(), "NaN in future covariates (spend/petrol beyond last known?)"
    return TimeSeries.from_dataframe(cal, time_col=time_col, value_cols=fut, freq=roles["freq"]).astype(np.float32)

In [ ]:
def iter_series(roles, vocab, input_len):
    """Yield (key, TimeSeries) holding exactly the last `input_len` days ending on ORIGIN."""
    import glob
    time_col, group_col, target_col = roles["time_col"], roles["group_col"], roles["target_col"]
    static = roles["static_covariates"]
    code_maps = {c: {v: i for i, v in enumerate(vocab[c])} for c in static}
    first_needed = ORIGIN - pd.Timedelta(days=input_len - 1)
    files = sorted(glob.glob(os.path.join(ACTUALS_DIR, "chunk_*.parquet")))
    assert files, f"no chunk_*.parquet in {ACTUALS_DIR}"
    for path in files:
        df = pd.read_parquet(path)
        df[time_col] = pd.to_datetime(df[time_col])
        df = df[(df[time_col] >= first_needed) & (df[time_col] <= ORIGIN)]
        for key, g in df.groupby(group_col, sort=False):
            g = g.sort_values(time_col)
            t = g[time_col]
            assert len(g) == input_len, f"{key}: {len(g)} days, need {input_len}"
            assert t.iloc[0] == first_needed and t.iloc[-1] == ORIGIN, \
                f"{key}: spans {t.iloc[0].date()} -> {t.iloc[-1].date()}, need {first_needed.date()} -> {ORIGIN.date()}"
            assert (t.diff().dropna() == pd.Timedelta(days=1)).all(), f"{key}: gaps in dates"
            y = g[target_col].to_numpy(dtype=np.float32)
            assert np.isfinite(y).all() and (y >= 0).all(), f"{key}: bad target values"
            codes = [code_maps[c].get(str(g[c].iloc[0]), 0) for c in static]   # 0 = __UNKNOWN__
            ts = TimeSeries.from_times_and_values(
                pd.DatetimeIndex(t), y.reshape(-1, 1), columns=[target_col],
                static_covariates=pd.DataFrame([codes], columns=static, dtype=np.float32))
            yield str(key), ts
        del df
        gc.collect()

## 4. Predict

In [ ]:
def predict_batch(model, batch_series, cov, n, num_samples):
    preds = model.predict(
        n=n, series=batch_series, future_covariates=[cov] * len(batch_series),
        num_samples=num_samples, verbose=False)
    if not isinstance(preds, list):
        preds = [preds]
    out = []
    for p in preds:
        v = p.all_values(copy=False)[:, 0, :]               # (n, samples)
        out.append((p.time_index, v.mean(axis=1), np.quantile(v, QUANTILES, axis=1)))
    return out

In [ ]:
def run(model=None):
    os.makedirs(OUT_DIR, exist_ok=True)
    roles, vocab = load_roles()
    input_len = roles["input_chunk_length"]
    out_len = roles["output_chunk_length"]
    n = (FORECAST_END - ORIGIN).days
    assert 0 < n <= out_len, f"n={n} must be <= output_chunk_length={out_len}; move the origin later"
    print(f"origin {ORIGIN.date()} | predict {(ORIGIN + pd.Timedelta(days=1)).date()} -> {FORECAST_END.date()} (n={n}) "
          f"| keep from {KEEP_FROM.date()}")

    cov = load_calendar(roles, input_len, n)
    model = model or load_model()

    rows, batch, keys, done = [], [], [], 0

    def flush():
        nonlocal rows, batch, keys, done
        if not batch:
            return
        for k, (idx, mean, q) in zip(keys, predict_batch(model, batch, cov, n, NUM_SAMPLES)):
            d = pd.DataFrame({roles["group_col"]: k, roles["time_col"]: idx, "PRED_MEAN": mean})
            for qi, qq in enumerate(QUANTILES):
                d[f"PRED_Q{int(qq * 100)}"] = q[qi]
            rows.append(d)
        done += len(batch)
        print(f"  predicted {done:,} series")
        batch, keys = [], []

    for key, ts in iter_series(roles, vocab, input_len):
        batch.append(ts)
        keys.append(key)
        if len(batch) == SERIES_BATCH:
            flush()
    flush()

    full = pd.concat(rows, ignore_index=True)
    assert full["PRED_MEAN"].notna().all(), "NaN in predictions"
    full.to_parquet(os.path.join(OUT_DIR, "pred_full_horizon.parquet"), index=False)
    keep = full[full[roles["time_col"]] >= KEEP_FROM]
    keep.to_parquet(os.path.join(OUT_DIR, "pred_dec11_dec31.parquet"), index=False)

    daily = full.groupby(roles["time_col"])["PRED_MEAN"].sum()
    daily.to_csv(os.path.join(OUT_DIR, "daily_total_mean.csv"))
    print("\nDec 11-31 total (mean):", round(keep["PRED_MEAN"].sum()))
    print("Daily total, last 10 days:\n", daily.tail(10).round(0).to_string())
    return full, keep

## 5. Run

In [ ]:
full, keep = run()

## 6. Sanity checks
Compare the days that overlap your earlier Aug 31-origin forecast (Nov 2 → Dec 10). Large gaps are the signal to investigate.

In [ ]:
daily = full.groupby(full.columns[1])['PRED_MEAN'].sum()
print(daily.describe().round(0))
print('Dec 11-31 total (mean):', round(keep['PRED_MEAN'].sum()))
# optional: compare with the earlier run
# old = pd.read_parquet(r'<path to earlier Aug-31-origin forecast>')   # needs same key/date/PRED_MEAN columns
# cmp = full.merge(old, on=[full.columns[0], full.columns[1]], suffixes=('_new','_old'))
# print(cmp.groupby(full.columns[1])[['PRED_MEAN_new','PRED_MEAN_old']].sum().round(0))